<a href="https://colab.research.google.com/github/gutivanian/API_Ecommerce_Midtrans/blob/master/agents/gemini_data_analytics/intro_gemini_data_analytics_http.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Intro to Gemini Data Analytics

| Author |
| --- |
| [Aditya Verma](https://github.com/vermaAstra) |

# Background and Overview
The **Conversational Analytics API** lets you chat with your BigQuery or Looker data anywhere, including embedded Looker dashboards, Slack and other chat apps, or even your own web applications. Your team members can get answers where they need them, when they need them, in the applications they use every day. You can find the [Colab example with the Python SDK here](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/gemini_data_analytics/intro_gemini_data_analytics_sdk.ipynb). This is a **Pre-GA** product. See [documentation](https://cloud.google.com/gemini/docs/conversational-analytics-api/overview) for more details.

Please provide feedback to conversational-analytics-api-feedback@google.com
<br>
### This notebook will help you
1. Authenticate to Google Cloud
2. Add data
3. Perform agent operations (create, list, get, delete)
4. Manage conversations (create, list, get, delete)
5. Ask questions with your agent


# Get Started

## API Enablement

**Please fill in the billing_project form field with your own Google Cloud project.  The project must have the following APIs enabled:**
-  [cloudaicompanion API](https://console.cloud.google.com/apis/library/cloudaicompanion.googleapis.com)
-  [Gemini Data Analytics API](https://console.cloud.google.com/apis/library/geminidataanalytics.googleapis.com)
-  [BQ API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com)
-  [Dataform API](https://console.cloud.google.com/apis/library/dataform.googleapis.com)
- [Agent Platform API](https://console.cloud.google.com/apis/library/aiplatform.googleapis.com)

You may pass in any BigQuery project/dataset/table for which you have read permissions.


## Auth and Imports

In [30]:
import json
import json as json_lib
import textwrap

import altair as alt
from google.colab import auth
from IPython.display import HTML, display
import pandas as pd
from pygments import formatters, highlight, lexers
import requests

auth.authenticate_user()

access_token = !gcloud auth application-default print-access-token
headers = {
    "Authorization": f"Bearer {access_token[0]}",
    "Content-Type": "application/json",
}

# Datasource and Billing project


*   Define your billing project within Google Cloud
*   Link your agent to data sources (BigQuery, Looker, Looker Studio)


In [31]:
# @title Billing Project and Prompt (System Instruction)

# fmt: off
billing_project = "project-4edfdecd-6192-4f0d-be4"  # @param {type:"string"}

location = "global"
api_version = "v1beta"

# provide critical context for your Conversational Analytics Agent here
system_instruction = "Think like an Analyst"  # @param {type:"string"}
# fmt: on

bigquery_data_sources = {}
looker_data_sources = {}
looker_studio_data_sources = {}

In [32]:
# @title BigQuery Datasource

bigquery_data_sources = {
    "bq": {
        "tableReferences": [
            {
                "projectId": "project-4edfdecd-6192-4f0d-be4",
                "datasetId": "ecommerce",
                "tableId": "distribution_centers",
            },
            {
                "projectId": "project-4edfdecd-6192-4f0d-be4",
                "datasetId": "ecommerce",
                "tableId": "events",
            },
            {
                "projectId": "project-4edfdecd-6192-4f0d-be4",
                "datasetId": "ecommerce",
                "tableId": "inventory_items",
            },
            {
                "projectId": "project-4edfdecd-6192-4f0d-be4",
                "datasetId": "ecommerce",
                "tableId": "order_items",
            },
            {
                "projectId": "project-4edfdecd-6192-4f0d-be4",
                "datasetId": "ecommerce",
                "tableId": "orders",
            },
            {
                "projectId": "project-4edfdecd-6192-4f0d-be4",
                "datasetId": "ecommerce",
                "tableId": "products",
            },
            {
                "projectId": "project-4edfdecd-6192-4f0d-be4",
                "datasetId": "ecommerce",
                "tableId": "users",
            },
            # Add more table references here
        ]
    }
}

# optional example queries (only leveraged for BigQuery datasources currently)
# Example queries can be parameterized in which case parameters must be defined.
# Each parameter requires a name (exactly as it appears in the question and SQL),
# description, and a valid dataType
# (https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/data-types#data_type_list).
example_queries = [
    {
        "naturalLanguageQuestion": (
            "What is the total sales revenue generated by each product category?"
        ),
        "sqlQuery": "SELECT t2.product_category, SUM(t1.sale_price) AS total_revenue FROM `project-4edfdecd-6192-4f0d-be4.ecommerce.order_items` AS t1 JOIN `project-4edfdecd-6192-4f0d-be4.ecommerce.inventory_items` AS t2 ON t1.inventory_item_id = t2.id GROUP BY t2.product_category ORDER BY total_revenue DESC;",
    },
]

glossary_terms = [
    {
        "display_name": "revenue",
        "description": ("total sales"        ),
        "labels": ["rev"],
    },
]

use_glossary_terms = True  # @param {type: "boolean"}
use_example_queries = True  # @param {type:"boolean"}

In [38]:
# @title Select datasource

selected_datasource = "bigquery_data_sources"  # @param ["bigquery_data_sources", "looker_data_sources", "looker_studio_data_sources"]

datasource_map = {
    "bigquery_data_sources": bigquery_data_sources,
}

datasource_references = datasource_map[selected_datasource]

bq_selected = False

if selected_datasource == "looker_data_sources":
  looker_selected = True
elif selected_datasource == "bigquery_data_sources":
  bq_selected = True
elif selected_datasource == "looker_studio_data_sources":
  looker_studio_selected = True

# Environment

In [39]:
# fmt: off
environment = "prod"  # @param ["prod", "staging", "autopush"]
# fmt: on

base_url = "https://geminidataanalytics.googleapis.com"

if environment == "autopush":
  base_url = "https://autopush-geminidataanalytics.sandbox.googleapis.com"
elif environment == "staging":
  base_url = "https://staging-geminidataanalytics.sandbox.googleapis.com"
else:
  base_url = "https://geminidataanalytics.googleapis.com"

# Data Agent Services

In [40]:
# @title Create Data Agent

# fmt: off
data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
# fmt: on

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents"

data_agent_payload = {
    "name": (
        f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
    ),  # Optional
    "description": "This is the description of data_agent.",  # Optional
    "data_analytics_agent": {
        "published_context": {
            "datasource_references": datasource_references,
            "system_instruction": system_instruction,
            "options": {
                "analysis": {
                    # Optional - if wanting to use advanced analysis with python.
                    # Default is False.
                    "python": {"enabled": False}
                }
            },
        }
    },
}

if bq_selected:
  if use_example_queries:
    data_agent_payload["data_analytics_agent"]["published_context"][
        "example_queries"
    ] = example_queries
  if use_glossary_terms:
    data_agent_payload["data_analytics_agent"]["published_context"][
        "glossary_terms"
    ] = glossary_terms
elif looker_selected and use_looker_golden_queries:
  data_agent_payload["data_analytics_agent"]["published_context"][
      "looker_golden_queries"
  ] = looker_golden_queries

params = {"data_agent_id": data_agent_id}  # Optional

data_agent_response = requests.post(
    data_agent_url, params=params, json=data_agent_payload, headers=headers
)

if data_agent_response.status_code == 200:
  print("Data Agent created successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error creating Data Agent: {data_agent_response.status_code}")
  print(data_agent_response.text)

Error creating Data Agent: 409
{
  "error": {
    "code": 409,
    "message": "Resource 'projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd' already exists",
    "status": "ALREADY_EXISTS",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ResourceInfo",
        "resourceName": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd"
      }
    ]
  }
}



In [41]:
# @title Get Data Agent

# fmt: off
data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
# fmt: on

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"

data_agent_response = requests.get(data_agent_url, headers=headers)

if data_agent_response.status_code == 200:
  print("Fetched Data Agent successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error: {data_agent_response.status_code}")
  print(data_agent_response.text)

Fetched Data Agent successfully!
{
  "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd",
  "displayName": "test ecommerce",
  "description": "This is the latest description of data_agent.",
  "labels": {
    "published_context": "bq"
  },
  "createTime": "2026-05-26T07:57:47.474954690Z",
  "updateTime": "2026-05-26T08:25:49.745038889Z",
  "dataAnalyticsAgent": {
    "publishedContext": {
      "systemInstruction": "Think like an Analyst",
      "datasourceReferences": {
        "bq": {
          "tableReferences": [
            {
              "projectId": "project-4edfdecd-6192-4f0d-be4",
              "datasetId": "ecommerce",
              "tableId": "distribution_centers"
            },
            {
              "projectId": "project-4edfdecd-6192-4f0d-be4",
              "datasetId": "ecommerce",
              "tableId": "events"
            },
            {
              "projectId": "project-4edfdecd-6192-4f

In [ ]:
# @title List Data Agents

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents"

data_agent_response = requests.get(data_agent_url, headers=headers)

if data_agent_response.status_code == 200:
  print("Data Agents Listed successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error Listing Data Agents: {data_agent_response.status_code}")
  print(data_agent_response.text)

In [11]:
# @title List Accessible Data Agents

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents:listAccessible"

# Choose the desired creator filter
# fmt: off
creator_filter = "NONE"  # @param ["NONE", "CREATOR_ONLY", "NOT_CREATOR_ONLY"]
# fmt: on

# Add the creator_filter as a query parameter
params = {"creator_filter": creator_filter}

data_agent_response = requests.get(
    data_agent_url, headers=headers, params=params
)

if data_agent_response.status_code == 200:
  print("Data Agents Listed successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error Listing Data Agents: {data_agent_response.status_code}")
  print(data_agent_response.text)

Data Agents Listed successfully!
{
  "dataAgents": [
    {
      "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_ac33f6bf-d236-4079-96a3-a515f3f9fd49",
      "displayName": "TPC-DS Retail Insights",
      "description": "Expert retail analyst for the TPC-DS 1G dataset.\n\nExample questions: (1) Find customers who have returned items more than 20% more often than the average customer returns for a store in TN for the year 2000.\n(2) Report the total extended sales price per item brand of manufacturer 128 for all sales in November (month 11).\n(3) List all the states with at least 10 customers who during January 2001 bought items with the price tag at least 20% higher than the average price of items in the same category.",
      "labels": {
        "readonly": "true",
        "sample-agent-signature": "sample_agent_tpc_ds_v1"
      },
      "createTime": "2026-05-26T07:53:50.690425803Z",
      "updateTime": "2026-05-26T07:53:51.376230511Z",
      "dataA

In [12]:
# @title Update Data Agent

# fmt: off
data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
# fmt: on

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"

payload = {
    "description": "This is the latest description of data_agent.",
    "data_analytics_agent": {
        "published_context": {
            "datasource_references": datasource_references,
            "system_instruction": system_instruction,
        }
    },
}

fields = ["description", "data_analytics_agent"]
params = {"updateMask": ",".join(fields)}

data_agent_response = requests.patch(
    data_agent_url, headers=headers, params=params, json=payload
)

if data_agent_response.status_code == 200:
  print("Data Agent updated successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error Updating Data Agent: {data_agent_response.status_code}")
  print(data_agent_response.text)

Data Agent updated successfully!
{
  "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/operations/operation-1779783948996-652b43e0e9144-a29f8dff-5f3db8c2",
  "metadata": {
    "@type": "type.googleapis.com/google.cloud.geminidataanalytics.v1beta.OperationMetadata",
    "createTime": "2026-05-26T08:25:49.417760881Z",
    "target": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd",
    "verb": "update",
    "requestedCancellation": false,
    "apiVersion": "v1beta"
  },
  "done": false
}


In [15]:
# @title [Agent Sharing] Set IAM Policy for Data Agent


"""PLEASE NOTE THIS API CALLS OVERRIDES EXISTING PERMISSION FOR THE RESOURCE.

For preserving existing policy in practise call Get IAM policy to fetch existing
policy and pass it along with additional changes in the call to Set IAM Policy
"""

# fmt: off
data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
users = "gutivanaliefs@gmail.com"  # @param {type:"string"}
# fmt: on

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}:setIamPolicy"


# Request body
payload = {
    "policy": {
        "bindings": [{
            "role": "roles/geminidataanalytics.dataAgentEditor",
            "members": [f"user:{i.strip()}" for i in users.split(",")],
        }]
    }
}

data_agent_response = requests.post(
    data_agent_url, headers=headers, json=payload
)

if data_agent_response.status_code == 200:
  print("IAM Policy set successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error setting IAM policy: {data_agent_response.status_code}")
  print(data_agent_response.text)

IAM Policy set successfully!
{
  "version": 1,
  "etag": "BwZStEEa/2Q=",
  "bindings": [
    {
      "role": "roles/geminidataanalytics.dataAgentEditor",
      "members": [
        "user:Gutivanaliefs@gmail.com"
      ]
    }
  ]
}


In [16]:
# @title [Agent Sharing] Get IAM Policy for Data Agent

# fmt: off
data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
# fmt: on

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}:getIamPolicy"

# Request body
payload = {
    "resource": (
        f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
    )
}

data_agent_response = requests.post(
    data_agent_url, headers=headers, json=payload
)

if data_agent_response.status_code == 200:
  print("IAM Policy fetched successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error fetching IAM policy: {data_agent_response.status_code}")
  print(data_agent_response.text)

IAM Policy fetched successfully!
{
  "version": 1,
  "etag": "BwZStEEa/2Q=",
  "bindings": [
    {
      "role": "roles/geminidataanalytics.dataAgentEditor",
      "members": [
        "user:Gutivanaliefs@gmail.com"
      ]
    }
  ]
}


In [ ]:
# @title [Soft Delete] Delete Data Agent

# fmt: off
data_agent_id = "data_agent_1"  # @param {type:"string"}
# fmt: on

data_agent_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"

data_agent_response = requests.delete(data_agent_url, headers=headers)

if data_agent_response.status_code == 200:
  print("Data Agent deleted successfully!")
  print(json.dumps(data_agent_response.json(), indent=2))
else:
  print(f"Error Deleting Data Agent: {data_agent_response.status_code}")
  print(data_agent_response.text)

# Data Chat Service

In [17]:
# @title Create Conversation

# fmt: off
data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
conversation_id = "conversation_1"  # @param {type:"string"}
# fmt: on

conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"

conversation_payload = {
    "agents": [
        f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
    ],
    "name": (
        f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}"
    ),
}
params = {"conversation_id": conversation_id}

conversation_response = requests.post(
    conversation_url, headers=headers, params=params, json=conversation_payload
)

if conversation_response.status_code == 200:
  print("Conversation created successfully!")
  print(json.dumps(conversation_response.json(), indent=2))
else:
  print(f"Error creating Conversation: {conversation_response.status_code}")
  print(conversation_response.text)

Conversation created successfully!
{
  "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/conversations/conversation_1",
  "agents": [
    "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd"
  ],
  "createTime": "2026-05-26T08:27:07.670170Z",
  "lastUsedTime": "2026-05-26T08:27:07.936004Z"
}


In [18]:
# @title Get Conversation

# fmt: off
conversation_id = "conversation_1"  # @param {type:"string"}
# fmt: on

conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations/{conversation_id}"

conversation_response = requests.get(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
  print("Conversation fetched successfully!")
  print(json.dumps(conversation_response.json(), indent=2))
else:
  print(
      f"Error while fetching conversation: {conversation_response.status_code}"
  )
  print(conversation_response.text)

Conversation fetched successfully!
{
  "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/conversations/conversation_1",
  "agents": [
    "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd"
  ],
  "createTime": "2026-05-26T08:27:07.670170Z",
  "lastUsedTime": "2026-05-26T08:27:07.936004Z"
}


In [19]:
# @title List Conversation

conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations"


conversation_response = requests.get(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
  print("Conversation fetched successfully!")
  print(json.dumps(conversation_response.json(), indent=2))
else:
  print(
      f"Error while fetching conversation: {conversation_response.status_code}"
  )
  print(conversation_response.text)

Conversation fetched successfully!
{
  "conversations": [
    {
      "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/conversations/conversation_1",
      "agents": [
        "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd"
      ],
      "createTime": "2026-05-26T08:27:07.670170Z",
      "lastUsedTime": "2026-05-26T08:27:07.936004Z"
    },
    {
      "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/conversations/ccea0272-d367-42b8-a1e0-a2a1b9bed211",
      "agents": [
        "projects/project-4edfdecd-6192-4f0d-be4/locations/global/dataAgents/agent_2b34caf2-fe71-432b-885d-545f2b934ddd"
      ],
      "createTime": "2026-05-26T07:59:06.795473Z",
      "lastUsedTime": "2026-05-26T08:03:55.340Z",
      "labels": {
        "host": "bigquery"
      }
    },
    {
      "name": "projects/project-4edfdecd-6192-4f0d-be4/locations/global/conversations/1572dff6-c309-4665-a0d5-f8e504104f40",
    

In [ ]:
# @title Delete Conversation

# fmt: off
conversation_id = "conversation_1"  # @param {type:"string"}
# fmt: on

conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations/{conversation_id}"

conversation_response = requests.delete(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
  print("Conversation deleted successfully!")
  print(json.dumps(conversation_response.json(), indent=2))
else:
  print(
      f"Error while deleting conversation: {conversation_response.status_code}"
  )
  print(conversation_response.text)

In [20]:
# @title Streaming Chat Messages


def is_json(str):
  try:
    json_object = json_lib.loads(str)
  except ValueError:
    return False
  return True


def handle_text_response(resp):
  parts = resp["parts"]
  full_text = "".join(parts)
  if "\n" not in full_text and len(full_text) > 80:
    wrapped_text = textwrap.fill(full_text, width=80)
    print(wrapped_text)
  else:
    print(full_text)


def get_property(data, field_name, default=""):
  return data[field_name] if field_name in data else default


def display_schema(data):
  fields = data["fields"]
  df = pd.DataFrame({
      "Column": map(lambda field: get_property(field, "name"), fields),
      "Type": map(lambda field: get_property(field, "type"), fields),
      "Description": map(
          lambda field: get_property(field, "description", "-"), fields
      ),
      "Mode": map(lambda field: get_property(field, "mode"), fields),
  })
  display(df)


def display_section_title(text):
  display(HTML(f"<h2>{text}</h2>"))


def format_bq_table_ref(table_ref):
  return "{}.{}.{}".format(
      table_ref["projectId"], table_ref["datasetId"], table_ref["tableId"]
  )


def format_looker_table_ref(table_ref):
  return "lookmlModel: {}, explore: {}".format(
      table_ref["lookmlModel"], table_ref["explore"]
  )


def display_datasource(datasource):
  source_name = ""

  if "studioDatasourceId" in datasource:
    source_name = datasource["studioDatasourceId"]
  elif "lookerExploreReference" in datasource:
    source_name = format_looker_table_ref(datasource["lookerExploreReference"])
  else:
    source_name = format_bq_table_ref(datasource["bigqueryTableReference"])

  print(source_name)
  if "schema" in datasource:
    display_schema(datasource["schema"])


def handle_schema_response(resp):
  if "query" in resp and "question" in resp["query"]:
    print(resp["query"]["question"])
  elif "result" in resp:
    display_section_title("Schema resolved")
    print("Data sources:")
    for datasource in resp["result"]["datasources"]:
      display_datasource(datasource)


def handle_data_response(resp):
  if "query" in resp:
    query = resp["query"]
    display_section_title("Retrieval query")
    if "name" in query:
      print("Query name: {}".format(query["name"]))
    if "question" in query:
      print("Question: {}".format(query["question"]))
    if "datasources" in query:
      print("Data sources:")
      for datasource in query["datasources"]:
        display_datasource(datasource)
  elif "generatedSql" in resp:
    display_section_title("SQL generated")
    print(resp["generatedSql"])
  elif "result" in resp:
    display_section_title("Data retrieved")

    fields = map(
        lambda field: get_property(field, "name"),
        resp["result"]["schema"]["fields"],
    )
    dict = {}

    for field in fields:
      dict[field] = [get_property(el, field) for el in resp["result"]["data"]]

    display(pd.DataFrame(dict))


def handle_chart_response(resp):
  if "query" in resp and "instructions" in resp["query"]:
    print(resp["query"]["instructions"])
  elif "result" in resp:
    vegaConfig = resp["result"]["vegaConfig"]
    alt.Chart.from_json(json_lib.dumps(vegaConfig)).display()


def handle_error(resp):
  display_section_title("Error")
  print("Code: {}".format(resp["code"]))
  print("Message: {}".format(resp["message"]))


def get_stream(url, json):
  s = requests.Session()

  acc = ""

  with s.post(url, json=json, headers=headers, stream=True) as resp:
    for line in resp.iter_lines():
      if not line:
        continue

      decoded_line = str(line, encoding="utf-8")

      if decoded_line == "[{":
        acc = "{"
      elif decoded_line == "}]":
        acc += "}"
      elif decoded_line == ",":
        continue
      else:
        acc += decoded_line

      if not is_json(acc):
        continue

      data_json = json_lib.loads(acc)

      if "systemMessage" not in data_json:
        if "error" in data_json:
          handle_error(data_json["error"])
        continue

      if "text" in data_json["systemMessage"]:
        handle_text_response(data_json["systemMessage"]["text"])
      elif "schema" in data_json["systemMessage"]:
        handle_schema_response(data_json["systemMessage"]["schema"])
      elif "data" in data_json["systemMessage"]:
        handle_data_response(data_json["systemMessage"]["data"])
      elif "chart" in data_json["systemMessage"]:
        handle_chart_response(data_json["systemMessage"]["chart"])
      else:
        colored_json = highlight(
            acc, lexers.JsonLexer(), formatters.TerminalFormatter()
        )
        print(colored_json)
      print("\n")
      acc = ""

In [23]:
# @title [Stateful] Chat using Conversation

data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
conversation_id = "conversation_1"  # @param {type:"string"}

# fmt: off
question = "Make a bar graph for the top 5 product, **Total Revenue** - Ranked by the sum of sales prices."  # @param {type:"string"}
# fmt: on

chat_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}:chat"

# Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [{"userMessage": {"text": question}}],
    "conversation_reference": {
        "conversation": (
            f"projects/{billing_project}/locations/{location}/conversations/{conversation_id}"
        ),
        "data_agent_context": {
            "data_agent": (
                f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
            ),
        },
    },
}
# Call get_stream method to stream the response
get_stream(chat_url, chat_payload)

Analyzing contextRetrieved context for 7 tables.


My Thought Process for Generating the Top 5 Products by Revenue Bar
ChartAlright, let's break down how I arrived at the solution for your request to
visualize the top 5 products by total revenue. The core of your request is
clear: you want a bar graph showing the top 5 products based on their total
revenue, which you've defined as the sum of their sales prices. Looking at the
available schema, I see two key tables: `products`: This will give me the
`product_id` and the `product_name` – essential for labeling my chart.
`order_items`: This table contains the `product_id` and, crucially, the
`sale_price` for each item sold. My immediate thought is that I need to connect
these two pieces of information. This means I'll have to perform a `JOIN`
operation. Specifically, an `INNER JOIN` makes the most sense because I only
care about order items that actually correspond to a product in the `products`
table. The join condition will naturally be

Data sources:
project-4edfdecd-6192-4f0d-be4.ecommerce.order_items
project-4edfdecd-6192-4f0d-be4.ecommerce.products




SELECT
    products.id AS product_id,
    products.name AS product_name,
    SUM(order_items.sale_price) AS total_revenue
FROM
    `project-4edfdecd-6192-4f0d-be4.ecommerce.products` AS products
INNER JOIN
    `project-4edfdecd-6192-4f0d-be4.ecommerce.order_items` AS order_items
    ON products.id = order_items.product_id
GROUP BY
    product_id,
    product_name
ORDER BY
    total_revenue DESC
LIMIT 5;






,product_id,product_name,total_revenue
0,24110,Woolrich Arctic Parka DF,13860
1,23989,The North Face Freedom Mens Ski Pants 2013,12642
2,24899,Nike Jordan Retro 11 Bred Bootie Socks,9030
3,2557,Canada Goose Women's Mystique,9000
4,24201,Mens Nike AirJordan Varsity Hoodie Jacket Grey...,8127


Data sources:
project-4edfdecd-6192-4f0d-be4.ecommerce.order_items
project-4edfdecd-6192-4f0d-be4.ecommerce.products




SELECT
    products.id AS product_id,
    products.name AS product_name,
    SUM(order_items.sale_price) AS total_revenue
FROM
    `project-4edfdecd-6192-4f0d-be4.ecommerce.products` AS products
INNER JOIN
    `project-4edfdecd-6192-4f0d-be4.ecommerce.order_items` AS order_items
    ON products.id = order_items.product_id
GROUP BY
    product_id,
    product_name
ORDER BY
    total_revenue DESC
LIMIT 5;








alt.Chart(...)



### Insights

*   **Revenue Leader:** The **Woolrich Arctic Parka DF** is the highest-grossing product, generating **$13,860** in total revenue.
*   **High-Value Apparel:** Most of the top-performing products are high-value winter wear or sports apparel, such as the **The North Face Freedom Mens Ski Pants** and the **Canada Goose Women's Mystique** parka.
*   **Revenue Concentration:** These top 5 products alone have generated significant revenue, ranging from approximately $8,100 to nearly $14,000 each.


What are the top 5 products by total units sold?Which product brands generate
the most revenue?Predict the revenue for the top-selling product for the next 6
months.




In [29]:
# @title [Stateful] Chat using Data Agent

data_agent_id = "agent_2b34caf2-fe71-432b-885d-545f2b934ddd"  # @param {type:"string"}
# fmt: off
question = "Make a bar graph for the top 5 product, **Total Revenue** - Ranked by the sum of sales prices."  # @param {type:"string"}
# fmt: on

chat_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}:chat"

# Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [{"userMessage": {"text": question}}],
    "data_agent_context": {
        "data_agent": (
            f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
        ),
    },
}

# if looker_selected:
  # chat_payload["data_agent_context"]["credentials"] = looker_credentials

# Call get_stream method to stream the response
get_stream(chat_url, chat_payload)

Code: 401
Message: Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.


In [26]:
# @title [Stateless] Chat using Inline Context

chat_url = (
    f"{base_url}/{api_version}/projects/{billing_project}/locations/global:chat"
)

# fmt: off
question = "Make a bar graph for the top 5 product, **Total Revenue** - Ranked by the sum of sales prices."  # @param {type:"string"}
# fmt: on

datasource_refs = dict(datasource_references)

# use credentials in datasource_references only in stateless chat request using inline context
# if looker_selected:
#   datasource_refs["looker"] = {
#       **datasource_refs["looker"],
#       "credentials": looker_credentials,
#   }

# Construct the payload
chat_payload = {
    "parent": f"projects/{billing_project}/locations/global",
    "messages": [{"userMessage": {"text": question}}],
    "inline_context": {
        "datasource_references": datasource_refs,
        "options": {
            "analysis": {
                # Optional - if wanting to use advanced analysis with python.
                # Default is False.
                "python": {"enabled": False}
            }
        },
    },
}

if bq_selected:
  if use_example_queries:
    chat_payload["inline_context"]["example_queries"] = example_queries
  if use_glossary_terms:
    chat_payload["inline_context"]["glossary_terms"] = glossary_terms
elif looker_selected and use_looker_golden_queries:
  chat_payload["inline_context"][
      "looker_golden_queries"
  ] = looker_golden_queries

# Call get_stream method to stream the response
get_stream(chat_url, chat_payload)

Analyzing contextRetrieved context for 7 tables.


My Thought Process for Generating the Top 5 Products by Revenue Bar GraphThe
user's request is quite specific: they want a bar graph visualizing the top 5
products based on their total revenue. My first priority is to understand how
"total revenue" is defined. Consulting the provided glossary, I see that "Total
revenue" is explicitly defined as "the sum of sales prices." This is a crucial
piece of information. Next, I need to identify the data sources required to
calculate this. To get the `sale_price`, I know I'll need to look at the
`order_items` table. To identify the products themselves and get their names,
I'll need to join this with a table that contains product information. The
`products` table seems like the most direct fit for this, as it contains product
`id` and `name`. Alternatively, `inventory_items` might also have product
information, but `products` feels more aligned with the direct "product"
concept. So, my plan is to 

Data sources:
project-4edfdecd-6192-4f0d-be4.ecommerce.order_items
project-4edfdecd-6192-4f0d-be4.ecommerce.products




SELECT
    products.id AS product_id,
    products.name AS product_name,
    SUM(order_items.sale_price) AS total_revenue
FROM
    `project-4edfdecd-6192-4f0d-be4.ecommerce.order_items` AS order_items
INNER JOIN
    `project-4edfdecd-6192-4f0d-be4.ecommerce.products` AS products
    ON order_items.product_id = products.id
GROUP BY
    product_id,
    product_name
ORDER BY
    total_revenue DESC
LIMIT 5;






,product_id,product_name,total_revenue
0,24110,Woolrich Arctic Parka DF,13860
1,23989,The North Face Freedom Mens Ski Pants 2013,12642
2,24899,Nike Jordan Retro 11 Bred Bootie Socks,9030
3,2557,Canada Goose Women's Mystique,9000
4,24201,Mens Nike AirJordan Varsity Hoodie Jacket Grey...,8127


Data sources:
project-4edfdecd-6192-4f0d-be4.ecommerce.order_items
project-4edfdecd-6192-4f0d-be4.ecommerce.products




SELECT
    products.id AS product_id,
    products.name AS product_name,
    SUM(order_items.sale_price) AS total_revenue
FROM
    `project-4edfdecd-6192-4f0d-be4.ecommerce.order_items` AS order_items
INNER JOIN
    `project-4edfdecd-6192-4f0d-be4.ecommerce.products` AS products
    ON order_items.product_id = products.id
GROUP BY
    product_id,
    product_name
ORDER BY
    total_revenue DESC
LIMIT 5;








alt.Chart(...)



### Insights

*   **Leading Product**: The "Woolrich Arctic Parka DF" is the top-performing product, generating **$13,860.00** in total revenue.
*   **High-Value Items**: The list is dominated by premium outerwear and athletic brands, including "The North Face", "Nike", and "Canada Goose".
*   **Revenue Concentration**: There is a notable gap between the top two products and the rest of the top five, suggesting these specific winter items are significant drivers of sales value.


What are the top 5 brands by total revenue?Which product categories generate the
highest revenue?Predict the revenue for the top 5 products for the next quarter
based on historical trends.




In [27]:
# @title List Messages

# fmt: off
conversation_id = "conversation_1"  # @param {type:"string"}
# fmt: on

conversation_url = f"{base_url}/{api_version}/projects/{billing_project}/locations/{location}/conversations/{conversation_id}/messages"

conversation_response = requests.get(conversation_url, headers=headers)

# Handle the response
if conversation_response.status_code == 200:
  print("Conversation fetched successfully!")
  print(json.dumps(conversation_response.json(), indent=2))
else:
  print(
      f"Error while fetching conversation: {conversation_response.status_code}"
  )
  print(conversation_response.text)

Conversation fetched successfully!
{
  "messages": [
    {
      "messageId": "307c9695-a69e-4cd4-8b68-c2dbebfa8956",
      "message": {
        "timestamp": "2026-05-26T08:36:22.795003Z",
        "systemMessage": {
          "text": {
            "parts": [
              "What are the top 5 products by total units sold?",
              "Which product brands generate the most revenue?",
              "Predict the revenue for the top-selling product for the next 6 months."
            ],
            "textType": "FOLLOWUP_QUESTIONS"
          }
        }
      }
    },
    {
      "messageId": "35a0d689-8d73-42bb-9f30-367e332322df",
      "message": {
        "timestamp": "2026-05-26T08:36:22.000540Z",
        "systemMessage": {
          "text": {
            "parts": [
              "### Insights\n\n*   **Revenue Leader:** The **Woolrich Arctic Parka DF** is the highest-grossing product, generating **$13,860** in total revenue.\n*   **High-Value Apparel:** Most of the top-performing pr

## Multi-turn Conversation with Data Agent or Inline Context

In [28]:
# @title get_stream function for multi-turn conversation

# NOTE: This methods is same as get_stream() method present in Streaming Chat Messages section.
# The only difference is that - here we are storing the response in an array "conversation_messages" to save the conversation.


def get_stream_multi_turn(url, json, conversation_messages):
  s = requests.Session()

  acc = ""

  with s.post(url, json=json, headers=headers, stream=True) as resp:
    for line in resp.iter_lines():
      if not line:
        continue

      decoded_line = str(line, encoding="utf-8")

      if decoded_line == "[{":
        acc = "{"
      elif decoded_line == "}]":
        acc += "}"
      elif decoded_line == ",":
        continue
      else:
        acc += decoded_line

      if not is_json(acc):
        continue

      data_json = json_lib.loads(acc)
      # Store the response to be used in next iteration.
      conversation_messages.append(data_json)

      if "systemMessage" not in data_json:
        if "error" in data_json:
          handle_error(data_json["error"])
        continue

      if "text" in data_json["systemMessage"]:
        handle_text_response(data_json["systemMessage"]["text"])
      elif "schema" in data_json["systemMessage"]:
        handle_schema_response(data_json["systemMessage"]["schema"])
      elif "data" in data_json["systemMessage"]:
        handle_data_response(data_json["systemMessage"]["data"])
      elif "chart" in data_json["systemMessage"]:
        handle_chart_response(data_json["systemMessage"]["chart"])
      else:
        colored_json = highlight(
            acc, lexers.JsonLexer(), formatters.TerminalFormatter()
        )
        print(colored_json)
      print("\n")
      acc = ""

In [ ]:
# @title Multi-turn Conversation helper function
chat_url = (
    f"{base_url}/{api_version}/projects/{billing_project}/locations/global:chat"
)

# Re-used across requests to track previous turns
conversation_messages = []

# fmt: off
context = "data_agent_context" # @param ["data_agent_context", "inline_context"]
# Only for data_agent_context
data_agent_id = "data_agent_1"  # @param {type:"string"}
# fmt: on


# Helper function for calling the API
def multi_turn_conversation(msg):
  userMessage = {"userMessage": {"text": msg}}

  # Send multi-turn request by including previous turns, plus new message
  conversation_messages.append(userMessage)

  datasource_refs = dict(datasource_references)

  # Construct the payload
  chat_payload = (
      {
          "parent": f"projects/{billing_project}/locations/global",
          "messages": conversation_messages,
          "data_agent_context": {
              "data_agent": (
                  f"projects/{billing_project}/locations/{location}/dataAgents/{data_agent_id}"
              ),
          },
      }
      if context == "data_agent_context"
      else {
          "parent": f"projects/{billing_project}/locations/global",
          "messages": conversation_messages,
          "inline_context": {
              "datasource_references": datasource_refs,
          },
      }
  )

  if looker_selected:
    if context == "inline_context":
      datasource_refs["looker"] = {
          **datasource_refs["looker"],
          "credentials": looker_credentials,
      }
      if use_looker_golden_queries:
        chat_payload["inline_context"][
            "looker_golden_queries"
        ] = looker_golden_queries
    else:
      chat_payload["data_agent_context"]["credentials"] = looker_credentials
  elif bq_selected and context == "inline_context":
    if use_example_queries:
      chat_payload["inline_context"]["example_queries"] = example_queries
    if use_glossary_terms:
      chat_payload["inline_context"]["glossary_terms"] = glossary_terms

  # Call get_stream_multi_turn method to stream the response
  get_stream_multi_turn(chat_url, chat_payload, conversation_messages)

In [ ]:
# Send first-turn request
multi_turn_conversation(
    "List of the top 5 states by the total number of airports"
)

In [ ]:
# Send follow-up-turn request
multi_turn_conversation("Can you please make it a pie chart?")